# Reproducing Gillon, Lecoq et al. (2023) from dandiset 000037

Paper. Gillon CJ, Lecoq JA, Pina JE, et al. (2023). Responses of pyramidal cell somata and
apical dendrites in mouse visual cortex over multiple days. *Scientific Data* 10, 287.
[doi:10.1038/s41597-023-02214-y](https://doi.org/10.1038/s41597-023-02214-y). A data descriptor.
The claims file (`claims/000037/10.1038-s41597-023-02214-y.yaml`) was written from the version of
record as served by Europe PMC (PMC10192216), with Supplementary Table 1 read as metadata.csv of
the authors' metadata repository.

Dandiset. [DANDI:000037](https://dandiarchive.org/dandiset/000037/0.230426.0054), version
0.230426.0054. The paper cites this version: its reference 13 carries the DOI
10.48324/DANDI.000037/0.230426.0054 (in the Crossref record of the article; the Europe PMC text
prints the reference without it). The later version 0.240209.1623 holds the same 150 NWB files
with the same blobs, and adds a CHANGES file and links to the papers in its metadata (shown in
the data check). Only the basic files (`behavior+ophys`) are opened; each is streamed into memory
once and nothing is written to disk.

Outside inputs, all pinned and named in the helper `openscope_ca.py`: Supplementary Table 1
(metadata.csv of `jeromelecoq/allen_openscope_metadata` at commit 0e9c7ac), used for Tables 1 and
2 and for the local session numbers and QC results that select the paper's 33 analysed sessions;
and from the paper's repository `colleenjg/OpenScope_CA_Analysis` at commit 046e7fe, the ROI
matching files of the Fig. 8 example mice and the source data of Fig. 7a. Supplementary Table 1
agrees with Tables 1 and 2 of the paper text in every field (checked while preparing this entry,
not run here).

| Claim | Tested |
|---|---|
| table1-subjects, cohorts-by-plane, sex-split, sessions-total, table2-sessions | tested |
| qc-failures | blocked: no QC status in the dataset |
| session-schedule, imaging-depths, acquisition-rates | tested |
| roi-noise-rejection, roi-count-ranges | tested |
| tracked-roi-totals, tracked-sessions-overlap, fig8a-final-matches, fig8b-final-matches | tested |
| fig8a-conflicts, fig8b-conflicts (with the paper's ROI matching files) | tested |
| fig8-somata-less-order-dependent | blocked: per-ordering counts published for two mice only |
| gabor-sequence-structure, visual-flow-structure, session-stimulus-durations | tested |
| inconsistent-fractions, inconsistent-blocks | tested |
| fig5-example-responses | tested; blocked, the decision rule leaves the Gabor panel open (question asked) |
| fig5-consistent-vs-inconsistent | tested; blocked as it depends on fig5-example-responses |
| fig6-behaviour-all-mice | tested |
| fig7a-snr-above-1 | tested |
| fig7a-snr-stable, fig7b-signal-stable | tested; blocked, the criterion's readings disagree |
| fig7c-correlations-similar | tested; blocked, the criterion's readings disagree |
| fig7c-high-correlation-fraction | tested; blocked as it depends on fig7c-correlations-similar |
| fig7a-source-link (with the paper's Fig. 7a source data) | tested |
| fig9-onset-offset-transients | tested |
| file-sizes, download-33-files | tested |

The analysis streams about 25 GB and takes about seven minutes.

In [ ]:
import ast
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats

import openscope_ca as oc

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
# LOAD-CELL: stream the 50 basic files once each and reduce them to what the claims use.
started = time.time()
assets = oc.version_assets()
latest = oc.version_assets(oc.LATEST_VERSION)
supp = oc.supplementary_table()
supp["session_id"] = supp.session_id.astype(str)
supp["subject_id"] = supp.subject_id.astype(str)
basic = assets[assets.kind == "B"].reset_index(drop=True)
basic["url"] = oc.run_parallel(oc.s3_url, basic.asset_id.tolist(), workers=8)
basic = basic.merge(supp[["session_id", "local_session_n", "QC"]], left_on="session",
                    right_on="session_id", how="left")
# The paper's analyses use local sessions 1 to 3 that passed QC (Supplementary Table 1), as the
# claims file states; correlations are computed for those, Fig. 5 measures for their session 1.
basic["analysed"] = (basic.local_session_n <= 3) & (basic.QC == "passed")
order = np.argsort(basic["size"].to_numpy())[::-1]


def job(i):
    r = basic.iloc[i]
    return oc.analyse(r.url, masks=True, corr=bool(r.analysed),
                      fig5=bool(r.analysed and r.local_session_n == 1))


done = oc.run_parallel(job, order.tolist(), weights=basic["size"].to_numpy()[order] / 1e9,
                       workers=4, budget=3.0)
S, R = {}, {}
for k, i in enumerate(order):
    S[basic.session[i]], R[basic.session[i]] = done[k]
print(f"streamed {len(S)} basic files ({basic['size'].sum() / 1e9:.1f} GB) of dandiset "
      f"{oc.DANDISET_ID} version {oc.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Assets at", oc.DANDISET_VERSION, ":", len(assets), "; at", oc.LATEST_VERSION, ":", len(latest))
both = assets.merge(latest, on="path", how="outer", suffixes=("_pinned", "_latest"), indicator=True)
print("Paths only in the latest version:", both.path[both._merge == "right_only"].tolist())
print("Paths only in the pinned version:", both.path[both._merge == "left_only"].tolist())
same = both[both._merge == "both"]
print("Common paths with a different blob:", int((same.blob_pinned != same.blob_latest).sum()), "of", len(same))
print("\nAssets by file version (B basic, I with stimulus images, S with imaging stack):")
print(assets.kind.value_counts(dropna=False).to_string())
per = assets.dropna(subset=["kind"]).pivot_table(index="session", columns="kind", values="size", aggfunc="count")
print("Sessions:", len(per), "; sessions with all three versions:", int((per.fillna(0) == 1).all(axis=1).sum()))
print("Subject folders:", assets.path.str.split("/").str[0].nunique())

The two published versions hold the same 150 NWB files, with the same blobs; 0.240209.1623 adds
a CHANGES file and links to the two papers in its metadata. Nothing the analysis uses differs.

In [ ]:
rows = []
for sid, s in S.items():
    loc = ast.literal_eval(s["location"]) if s["location"].startswith("{") else {}
    start = pd.Timestamp(s["session_start_time"])
    rows.append({
        "session": sid, "subject": s["subject_id"], "identifier": s["identifier"],
        "start": s["session_start_time"][:25], "date_file": start.strftime("%Y%m%d"),
        # The same clock read as UTC (Supplementary Table 1 gives these clock values as UTC),
        # converted to the Allen Institute's time zone.
        "date_if_utc": start.tz_localize(None).tz_localize("UTC").tz_convert("America/Los_Angeles").strftime("%Y%m%d"),
        "start_naive": start.tz_localize(None),
        "area": loc.get("area"), "depth": int(loc.get("depth")), "genotype": s["genotype"],
        "sex": s["sex"], "age_days": int(s["age"].strip("PD")), "species": s["species"],
        "n_rois": s["dff_shape"][1], "n_frames": s["dff_shape"][0],
        "n_tracked": int(np.isfinite(s["tracking_id"]).sum()),
        "imaging_rate_stated": s["imaging_rate"], "fps_2p": 1 / np.median(np.diff(s["dff_t"])),
        "fps_run": 1 / np.median(np.diff(s["run_t"])), "fps_pupil": 1 / np.median(np.diff(s["pupil_t"])),
        "indicator": s["indicator"], "excitation_nm": s["excitation_lambda"],
        "mask_shape": s["mask_shape"][1:], "max_proj_shape": s["max_projection_shape"]})
sess = pd.DataFrame(rows).sort_values(["subject", "start_naive"]).set_index("session")
sess["line"] = sess.genotype.map(lambda g: "L2/3" if g.startswith("Cux2") else "L5" if g.startswith("Rbp4") else "?")
# Plane from the recorded depth: layer 1 (dendrites) at 20 to 75 um, cell body layers at 175 and 375 um (Methods).
sess["plane"] = np.where(sess.depth < 100, "D", "S")
sess["group"] = sess.line + "-" + sess.plane
sess["local_n"] = sess.groupby("subject").cumcount() + 1
sess = sess.join(supp.set_index("session_id")[["local_session_n", "QC", "imaging_plane", "imaging_cell_layer"]])
sess["analysed"] = (sess.local_n <= 3) & (sess.QC == "passed")
print("Subjects:", sess.subject.nunique(), "; sessions:", len(sess), "; species:", sess.species.unique().tolist())
print("Session id equals file identifier and the id in the file name:", bool((sess.index == sess.identifier).all()))
print("Chronological session number equals Supplementary Table 1's local session number:",
      bool((sess.local_n == sess.local_session_n).all()))
print("Plane from depth equals Supplementary Table 1's plane:",
      bool((sess.plane.map({"D": "distal_apical_dendrites", "S": "somata"}) == sess.imaging_plane).all()))
print("Layer from genotype equals Supplementary Table 1's layer:", bool((sess.line == sess.imaging_cell_layer).all()))
print("\nSessions per subject and group:")
print(sess.groupby(["group", "subject"]).size().to_string())
print("\nAnalysed sessions (local 1 to 3, passed QC):", int(sess.analysed.sum()), "from",
      sess[sess.analysed].subject.nunique(), "mice")
print(sess[["subject", "group", "start", "depth", "sex", "age_days", "n_rois", "n_tracked", "local_n", "QC"]].to_string())

The files name 13 subjects and 50 sessions. Session numbers counted in order of start time
within each subject equal the local session numbers of Supplementary Table 1, the plane read
from the recorded depth equals the table's plane, and the layer read from the genotype equals
its layer. The QC result is not in the files: the analysed sessions (local 1 to 3, passed QC)
are taken from Supplementary Table 1.

In [ ]:
print("Stated imaging rate (Hz):", sess.imaging_rate_stated.unique().tolist())
print("Two-photon rate from timestamps (Hz):", sess.fps_2p.min().round(3), "to", sess.fps_2p.max().round(3))
print("Running rate (Hz):", sess.fps_run.min().round(3), "to", sess.fps_run.max().round(3),
      "; pupil rate (Hz):", sess.fps_pupil.min().round(3), "to", sess.fps_pupil.max().round(3))
print("Frames per session:", sess.n_frames.min(), "to", sess.n_frames.max(),
      "; recording length (min):", round(sess.n_frames.min() / sess.fps_2p.max() / 60, 2), "to",
      round(sess.n_frames.max() / sess.fps_2p.min() / 60, 2))
print("Indicator:", sess.indicator.unique().tolist(), "; excitation (nm):", sess.excitation_nm.unique().tolist(),
      "; area:", sess.area.unique().tolist())
print("Mask shapes:", sess.mask_shape.astype(str).unique().tolist(), "; max projection:",
      sess.max_proj_shape.astype(str).unique().tolist())
storage = pd.Series({sid: "contiguous" if S[sid]["dff_storage"][0] is None else f"{S[sid]['dff_storage'][1]}, chunks {S[sid]['dff_storage'][0]}" for sid in sess.index})
print("dF/F storage by plane:", pd.crosstab(sess.plane, storage.str.split(",").str[0]).to_dict())
chunks = [S[sid]["dff_storage"][0] for sid in sess.index if S[sid]["dff_storage"][0] is not None]
print("Chunk shapes (frames x ROIs) of the compressed files:", min(c[0] for c in chunks), "to", max(c[0] for c in chunks),
      "frames by", min(c[1] for c in chunks), "to", max(c[1] for c in chunks), "ROIs")
units = pd.Series({k: {S[s][k] for s in S} for k in ("dff_unit", "run_unit", "pupil_unit", "pupil_x_unit")})
print("Stated units:", units.to_dict())
dff_ranges = pd.DataFrame({s: {"min": R[s]["roi"]["min"].min(), "median": R[s]["roi"]["median"].median(),
                               "max": R[s]["roi"]["max"].max()} for s in R}).T
print("dF/F over all ROIs and sessions: min", round(dff_ranges["min"].min(), 2), "; median of ROI medians",
      round(dff_ranges["median"].median(), 4), "; max", round(dff_ranges["max"].max(), 2))
run = pd.DataFrame({s: {"min": np.nanmin(S[s]["run_v"]), "p1": np.nanpercentile(S[s]["run_v"], 1),
                        "median": np.nanmedian(S[s]["run_v"]), "p99": np.nanpercentile(S[s]["run_v"], 99),
                        "max": np.nanmax(S[s]["run_v"]), "nan": np.isnan(S[s]["run_v"]).mean()} for s in S}).T
pup = pd.DataFrame({s: {"min": np.nanmin(S[s]["pupil_d"]), "median": np.nanmedian(S[s]["pupil_d"]),
                        "max": np.nanmax(S[s]["pupil_d"]), "nan": np.isnan(S[s]["pupil_d"]).mean(),
                        "x_max": np.nanmax(S[s]["pupil_x"]), "y_max": np.nanmax(S[s]["pupil_y"])} for s in S}).T
print("\nRunning velocity (cm/s) over sessions:")
print(run.describe().loc[["min", "50%", "max"]].round(3).to_string())
print("\nPupil diameter and position (mm) over sessions:")
print(pup.describe().loc[["min", "50%", "max"]].round(3).to_string())

The stated imaging rate (30 Hz) agrees with the timestamps (about 30.1 Hz), running is sampled
at 60 Hz and pupil at the imaging rate. Running velocity is stated in cm/s, and its values (a
median near a few cm/s, percentiles 1 and 99 within tens of cm/s) are consistent with that,
not shown by it. Pupil diameter is stated in mm, with medians under 1 mm; positions are within
the 6.528 mm image width the files give. dF/F is stated as "Normalized fluorescence (A.U.)";
ROI medians near 0 are consistent with dF/F.

In [ ]:
# Stimulus tables: counts by stimulus and condition, per session.
tab = []
for sid in sess.index:
    t = S[sid]["trials"]
    tab.append({"session": sid, "rows": len(t), "grey": int((t.stimulus_type == "grayscreen").sum()),
                "gabor_images": int((t.stimulus_type == "gabors").sum()),
                "U_images": int((t.gabor_frame == "U").sum()),
                "flow_right": int((t.stimulus_template_name == "visflow_right").sum()),
                "flow_left": int((t.stimulus_template_name == "visflow_left").sum()),
                "flow_reversed": int(((t.stimulus_type == "visflow") & (t.unexpected == 1)).sum()),
                "first_stimulus": t.stimulus_template_name[t.stimulus_type != "grayscreen"].iloc[0],
                "stim_span_min": (t.stop_time.max() - t.start_time.min()) / 60})
tab = pd.DataFrame(tab).set_index("session")
print(tab.describe().round(2).to_string())
print("\nFirst stimulus after grey:", tab.first_stimulus.value_counts().to_dict())
print("Trial table columns:", S[sess.index[0]]["trial_columns"])
print("Stimulus types:", sorted({x for s in S.values() for x in s["trials"].stimulus_type.unique()}),
      "; flow directions:", sorted({x for s in S.values() for x in s["trials"].main_flow_direction.unique() if x}))

Every session holds four grey screen periods around three stimulus periods (Gabor sequences,
and visual flow in each direction), in an order that varies between sessions. The trial table
has no stimulus seed column, and no QC field is in the files.

In [ ]:
# Clock check. The trial table gives each grey screen transition as a two-photon frame. For each
# session, the population mean of the robust-scaled dF/F changes across the six transitions; the
# same contrast is computed with the transitions shifted in time by 10 to 300 s (500 shifts, fixed
# seed), which keeps the response shape but breaks the alignment.
rng = np.random.default_rng(0)


def transition_contrast(pop, frames, fps):
    w = int(round(fps))
    vals = [abs(pop[f:f + w].mean() - pop[f - w:f].mean()) for f in frames if f - w >= 0 and f + w <= len(pop)]
    return np.mean(vals)


clock = []
for sid in sess.index:
    pop, fr, fps = R[sid]["popmean_scaled"], R[sid]["transitions"].frame.to_numpy(), R[sid]["fps"]
    true = transition_contrast(pop, fr, fps)
    shifts = rng.uniform(10, 300, 500) * rng.choice([-1, 1], 500)
    null = np.array([transition_contrast(pop, fr + int(round(d * fps)), fps) for d in shifts])
    clock.append({"session": sid, "true": true, "null_median": np.median(null), "null_p99": np.percentile(null, 99),
                  "rank": (null >= true).mean()})
clock = pd.DataFrame(clock).set_index("session")
print(clock.round(4).to_string())
print("Sessions whose true contrast exceeds every shifted one:", int((clock["rank"] == 0).sum()), "of", len(clock))
print("Sessions with the true contrast below the 99th percentile of the shifted ones:",
      clock.index[clock.true <= clock.null_p99].tolist())

The contrast at the true transitions is above every shifted contrast in 38 of 50 sessions,
and above the 99th percentile of the shifted ones in 48. In those 48 sessions the dF/F shares the
clock of the trial table. It is not established in two sessions: 777496949 (mouse 418779, 15
ROIs, rank 0.054) and 838633305 (440889 local session 6, which failed QC; Supplementary Table 1
notes the laser was set to 800 nm). The second is not among the 33 analysed sessions; the first
is, and its group's results below are reported with the others. Running and pupil
carry their own timestamps on the same time base; no stimulus contrast was used for them, so
their alignment is stated, not shown.

In [ ]:
# Recordings deposited twice: compare running traces, pupil traces, dF/F summaries and trial times
# of every pair of sessions.
sig = {}
for sid in sess.index:
    s = S[sid]
    sig[sid] = (np.round(s["run_v"][:5000], 6).tobytes(), np.round(np.nan_to_num(s["pupil_d"][:5000]), 6).tobytes(),
                np.round(R[sid]["roi"]["mean"].to_numpy(), 6).tobytes(), np.round(s["trials"].start_time.to_numpy(), 4).tobytes())
pairs = []
ids = list(sig)
for i in range(len(ids)):
    for j in range(i + 1, len(ids)):
        same = [a == b for a, b in zip(sig[ids[i]], sig[ids[j]])]
        if any(same):
            pairs.append((ids[i], ids[j], same))
print("Pairs of sessions sharing running, pupil, dF/F means or trial times:", pairs if pairs else "none")
print("Pairs with the same ROI count:",
      [(a, b) for a, b in zip(sess.index[:-1], sess.index[1:]) if sess.n_rois[a] == sess.n_rois[b]])

No two sessions share running or pupil traces, ROI means or trial times, so no recording is
deposited twice among the basic files.

## table1-subjects

> The dataset holds 13 mice whose subject ids, sex, date of birth, imaged cell type (L2/3 or
> L5 pyramidal) and imaging plane (somata or distal apical dendrites) are as listed in Table 1.

Table 1's values are taken from Supplementary Table 1 (metadata.csv, which holds the columns of
Tables 1 and 2). The files record sex and genotype (cell type through the line); they record
no date of birth and no plane, which are listed and also shown as values implied by the recorded
age and depth.

In [ ]:
t1 = supp.groupby("subject_id").agg(sex=("subject_sex", "first"), dob=("subject_date_of_birth", "first"),
                                    layer=("imaging_cell_layer", "first"), plane=("imaging_plane", "first"))
first = sess.groupby("subject").first()
dob_implied = (pd.to_datetime(first.date_file) - pd.to_timedelta(first.age_days, unit="D")).dt.strftime("%Y%m%d")
dob_implied_utc = (pd.to_datetime(first.date_if_utc) - pd.to_timedelta(first.age_days, unit="D")).dt.strftime("%Y%m%d")
cmp1 = pd.DataFrame({"sex_table": t1.sex, "sex_file": first.sex, "layer_table": t1.layer, "layer_file": first.line,
                     "dob_table": t1.dob, "dob_from_age": dob_implied, "dob_from_age_if_utc": dob_implied_utc,
                     "plane_table": t1.plane, "plane_from_depth": first.plane.map({"D": "distal_apical_dendrites", "S": "somata"})})
print(cmp1.to_string())
print("\nSubjects in Table 1:", len(t1), "; in the files:", sess.subject.nunique(),
      "; in both:", len(set(t1.index) & set(sess.subject)))
recorded = {"sex": (cmp1.sex_table != cmp1.sex_file).sum(), "cell type (genotype)": (cmp1.layer_table != cmp1.layer_file).sum()}
print("Disagreements in recorded fields:", {k: int(v) for k, v in recorded.items()}, "; total", int(sum(recorded.values())))
print("Not recorded in the files: date of birth, plane")
print("Implied by the recorded age (variant): date of birth agrees for", int((cmp1.dob_table == cmp1.dob_from_age).sum()),
      "of 13 (", int((cmp1.dob_table == cmp1.dob_from_age_if_utc).sum()), "if the start clock is read as UTC)")
print("Implied by the recorded depth (variant): plane agrees for", int((cmp1.plane_table == cmp1.plane_from_depth).sum()), "of 13")

**Result: reproduced.** All 13 subjects of Table 1 are in the dataset, and their recorded sex and
genotype (cell type) agree with Table 1 for all 13 (0 disagreements). The files record no date of
birth and no plane. The variants agree: the plane implied by the recorded depth matches for 13 of
13, and the date of birth implied by the recorded age matches for 12 of 13; for mouse 418779 it
is one day late, and matches when the start clock is read as UTC (see table2-sessions).

## cohorts-by-plane

> Thirteen mice were imaged, three L2/3 dendritic, three L2/3 somatic, four L5 dendritic and
> three L5 somatic, each with at least three imaging sessions.

In [ ]:
mice = sess.groupby("subject").agg(group=("group", "first"), sessions=("group", "size"))
print(mice.groupby("group").agg(mice=("sessions", "size"), fewest_sessions=("sessions", "min"),
                                most_sessions=("sessions", "max")).to_string())
print("Expected: L2/3-D 3, L2/3-S 3, L5-D 4, L5-S 3; mice with fewer than three sessions:",
      mice.index[mice.sessions < 3].tolist())

**Result: reproduced.** By line and plane the dataset holds 3 L2/3 dendritic, 3 L2/3 somatic, 4 L5
dendritic and 3 L5 somatic mice, and every mouse has at least three sessions (3 to 6).

## sex-split

> Seven of the mice are male and six female, and only one L2/3 dendritic mouse is female.

In [ ]:
mice["sex"] = sess.groupby("subject").sex.first()
print(pd.crosstab(mice.group, mice.sex, margins=True).to_string())

**Result: reproduced.** 7 mice are male and 6 female, and one L2/3 dendritic mouse is female.

## sessions-total

> The dataset holds 50 imaging sessions from 13 animals, each session in its own files grouped
> by subject.

In [ ]:
folders = assets.path.str.split("/").str[0]
nwb = assets[assets.kind.notna()]
print("Distinct session ids:", nwb.session.nunique(), "; subject folders:", folders[assets.kind.notna()].nunique())
print("Every file sits in the folder of its subject:",
      bool((nwb.path.str.split("/").str[0] == "sub-" + nwb.subject).all()))
print("Files per session:", nwb.groupby("session").size().value_counts().to_dict(),
      "; sessions per subject:", nwb.groupby("subject").session.nunique().value_counts().sort_index().to_dict())

**Result: reproduced.** The dataset holds 50 distinct sessions in 13 subject folders, three files
per session, each in the folder of its subject.

## table2-sessions

> The 50 sessions have the subject ids, session ids, imaging dates, imaging depths, ROI counts,
> tracked ROI counts, QC results and stimulus seeds listed in Table 2.

Table 2's values are taken from Supplementary Table 1. The date of a session is the date of
its start time as the file states it; the same clock read as UTC (as Supplementary Table 1
labels it) and converted to Pacific time is shown as a variant.

In [ ]:
t2 = supp.set_index("session_id")
tracked_table = t2.n_tracked_ROIs.clip(lower=0)            # -1 in metadata.csv is 0 in Table 2
cmp2 = pd.DataFrame({
    "subject": sess.subject == t2.subject_id.reindex(sess.index),
    "date": sess.date_file == t2.imaging_date.reindex(sess.index),
    "depth": sess.depth == t2.imaging_depth.reindex(sess.index),
    "n_rois": sess.n_rois == t2.n_ROIs.reindex(sess.index),
    "n_tracked": sess.n_tracked == tracked_table.reindex(sess.index)})
print("Sessions in Table 2:", len(t2), "; in the files:", len(sess), "; in both:", len(set(t2.index) & set(sess.index)))
print("Fields not recorded in the files: QC, stimulus seed")
print("Sessions agreeing, per field:", cmp2.sum().to_dict())
print("Sessions agreeing in every recorded field:", int(cmp2.all(axis=1).sum()), "of 50")
bad = cmp2[~cmp2.all(axis=1)]
print(pd.concat([bad, sess.loc[bad.index, ["subject", "start", "date_file", "date_if_utc", "depth"]],
                 t2.loc[bad.index, ["imaging_date", "imaging_UTC_date_time", "imaging_depth"]]], axis=1).to_string())
alt = cmp2.copy()
alt["date"] = sess.date_if_utc == t2.imaging_date.reindex(sess.index)
print("\nVariant, clock read as UTC: dates agree for", int(alt.date.sum()), "of 50; sessions agreeing in every field:",
      int(alt.all(axis=1).sum()))
print("Supplementary Table 1's 'UTC' time equals the clock time of the file's start for",
      int((sess.start_naive.dt.strftime("%Y%m%dT%H%M%S") == t2.imaging_UTC_date_time.reindex(sess.index)).sum()), "of 50")

**Result: not reproduced.** All 50 sessions of Table 2 are in the dataset, and subject, ROI count and
tracked ROI count agree for all 50. Only 43 of 50 agree in every recorded field, below the 45 the
criterion asks for partial reproduction. The three sessions of mouse 433414 are recorded at a depth
of 60 um where Table 2 gives 75. Four sessions (412933 local session 3 and the three of 418779)
start one day later in the files than the date in Table 2. Their start times are evening Pacific
times stated with a Pacific offset (for example 2018-10-19T00:18-07:00). Supplementary Table 1
gives the same clock values as UTC for all 50 sessions. Read as UTC, all 50 dates agree and 47 of
50 sessions agree in every field (the depth difference remains). The files do not record QC or the
stimulus seed.

## qc-failures

> Two animals had at least one session that failed the Allen Institute's QC thresholds, and
> some animals had more than three sessions, for example when an early session failed QC.

Blocked: the files hold no QC status, and the criterion blocks the claim when the dataset
records none. The counts from Supplementary Table 1 are shown for reference only.

In [ ]:
qc_fields = [c for c in S[sess.index[0]]["trial_columns"] + S[sess.index[0]]["subject_fields"] if "qc" in c.lower() or "pass" in c.lower()]
print("QC fields in the files:", qc_fields if qc_fields else "none")
failed = sess[sess.QC == "failed"].groupby("subject").local_n.apply(list)
print("Supplementary Table 1, failed sessions by mouse (local numbers):", failed.to_dict())
print("(a) animals with any failed session:", len(failed),
      "; (b) animals whose first three sessions failed:", int(failed.apply(lambda v: set([1, 2, 3]) <= set(v)).sum()))
print("Animals with more than three sessions in the files:", mice.index[mice.sessions > 3].tolist())

## session-schedule

> Each mouse had three 70-minute imaging sessions within three to six days, with at most one
> session per day.

The criterion's four combinations: sessions (i) local 1 to 3, or (ii) the sessions of the
paper's analyses (local 1 to 3 passing QC, or local 4 to 6 for mice 411400 and 412933); span (a)
days from first to third session (3 to 6 allowed), or (b) calendar days counted inclusively
(3 to 6 allowed). Dates are those of the start times as the files state them; the clock read
as UTC is a variant.

In [ ]:
def schedule(date_col):
    out = []
    for subj, g in sess.groupby("subject"):
        sets = {"i": g[g.local_n <= 3]}
        sets["ii"] = g[g.local_n >= 4] if subj in ("411400", "412933") else g[g.analysed]
        for k, sub in sets.items():
            d = pd.to_datetime(sub[date_col]).sort_values()
            span = (d.iloc[-1] - d.iloc[0]).days
            out.append({"subject": subj, "sessions": k, "dates": " ".join(sub[date_col]),
                        "span_a": span, "span_b": span + 1, "shared_date": d.duplicated().any(),
                        "all_dates_distinct_in_mouse": not pd.to_datetime(g[date_col]).duplicated().any()})
    return pd.DataFrame(out)


for date_col in ("date_file", "date_if_utc"):
    sch = schedule(date_col)
    print(f"Dates from {date_col}:")
    print(sch.to_string(index=False))
    for k in ("i", "ii"):
        for span in ("span_a", "span_b"):
            sub = sch[sch.sessions == k]
            ok = sub[span].between(3, 6) & ~sub.shared_date
            print(f"  sessions ({k}), span ({span[-1]}): mice meeting it {int(ok.sum())} of {len(sub)}; failing:",
                  sub.subject[~ok].tolist())
    print()
dur = pd.Series({sid: (S[sid]["trials"].stop_time.max() - S[sid]["trials"].start_time.min()) / 60 for sid in sess.index})
rec = sess.n_frames / sess.fps_2p / 60
print("Session duration from the stimulus table (min):", dur.min().round(2), "to", dur.max().round(2),
      "; two-photon recording (min):", rec.min().round(2), "to", rec.max().round(2), "; within 70 +/- 7:",
      bool(dur.between(63, 77).all() and rec.between(63, 77).all()))

**Result: reproduced.** Under combination (i)(b), local sessions 1 to 3 with calendar days counted
inclusively, all 13 mice have spans of 3 to 6 days with no two sessions on one date. This holds
with both readings of the dates. The other combinations fail for several mice: with the dates as stated, (i)(a) and (ii)(a)
give 2-day spans for 6 and 5 mice, and (ii)(b) gives 7 days for 411400. Sessions last 70.06 to
70.37 min by the stimulus table and 70.22 to 70.53 min by the imaging, within 70 min +/- 10%.

## imaging-depths

> Somata were imaged at 175 um (L2/3) and 375 um (L5) depth, and distal apical dendrites at 50
> to 75 um (L2/3) and 20 um (L5).

In [ ]:
allowed = {"L2/3-S": {175}, "L5-S": {375}, "L2/3-D": {50, 75}, "L5-D": {20}}
sess["depth_ok"] = [d in allowed[g] for d, g in zip(sess.depth, sess.group)]
print(sess.groupby(["group", "depth"]).size().rename("sessions").to_string())
print("Sessions whose depth equals the value for their line and plane:", int(sess.depth_ok.sum()), "of 50")
print("Sessions that do not:", sess[~sess.depth_ok][["subject", "group", "depth"]].to_dict("index"))
print("Sessions within the stated ranges (175, 375, 50 to 75, 20):",
      int(sum((g == "L2/3-D" and 50 <= d <= 75) or d in allowed[g] for d, g in zip(sess.depth, sess.group))), "of 50")

**Result: not reproduced.** 47 of 50 sessions are at the depth the criterion allows for their line
and plane (175 and 375 um for somata, 20 um for L5 dendrites, 50 or 75 um for L2/3 dendrites). The
three sessions of L2/3 dendritic mouse 433414 are at 60 um. That is within the 50 to 75 um range of
the statement, which all 50 sessions meet. The criterion admits only 50 or 75, and three
exceptions are one more than it allows for partial reproduction. Table 2 gives 75 um for these
sessions (table2-sessions), so the dataset and the paper disagree on this depth.

## acquisition-rates

> Calcium movies were recorded at 30 Hz over a 400 um field of view at 512 x 512 pixels, and
> running was recorded at 60 Hz in cm/s.

In [ ]:
print("Imaging rate from dF/F timestamps (Hz): min", sess.fps_2p.min().round(3), "max", sess.fps_2p.max().round(3),
      "; within 30 +/- 3 in", int(sess.fps_2p.between(27, 33).sum()), "of 50")
print("Running rate from timestamps (Hz): min", sess.fps_run.min().round(3), "max", sess.fps_run.max().round(3),
      "; within 60 +/- 6 in", int(sess.fps_run.between(54, 66).sum()), "of 50; unit", units["run_unit"])
fs = sess.mask_shape.apply(lambda s: s[0] == 512 and s[1] == 512) & sess.max_proj_shape.apply(lambda s: tuple(s) == (512, 512))
print("Frame size 512 x 512 (masks and max projection) in", int(fs.sum()), "of 50")
print("Field of view in um: not recorded (the max projection carries a 'resolution' attribute of 0.000128, no unit)")

**Result: reproduced.** Imaging runs at 30.08 to 30.15 Hz by the timestamps (stated 30 Hz), running
at 59.95 Hz in cm/s, and every mask and projection image is 512 x 512 pixels. The 400 um field of
view is not recorded in the files.

## roi-noise-rejection

> ROIs that were too noisy, with a mean dF/F below 0 or a median dF/F above the midpoint between
> the trace's minimum and maximum, were rejected, so no retained ROI has either property.

In [ ]:
noise = []
for sid in sess.index:
    r = R[sid]["roi"]
    mid = (r["min"] + r["max"]) / 2
    noise.append({"session": sid, "group": sess.group[sid], "rois": len(r), "mean_below_0": int((r["mean"] < 0).sum()),
                  "median_above_midrange": int((r["median"] > mid).sum()),
                  "either": int(((r["mean"] < 0) | (r["median"] > mid)).sum()),
                  "lowest_mean": r["mean"].min(), "highest_median_minus_mid": (r["median"] - mid).max()})
noise = pd.DataFrame(noise).set_index("session")
print(noise.to_string())
print("ROIs with either property:", noise.either.sum(), "of", noise.rois.sum(),
      f"({100 * noise.either.sum() / noise.rois.sum():.3f}%)", "; sessions with any:", int((noise.either > 0).sum()))

**Result: reproduced.** None of the 19115 ROIs in the 50 sessions has a mean dF/F below 0 or a median
above its mid-range. The lowest ROI mean is 0.0006, and every median is below its mid-range.

## roi-count-ranges

> Somatic planes yielded 15 to 224 ROIs per mouse per session and dendritic planes 159 to 1636.

In [ ]:
readings = {"(a) 33 analysed sessions": sess.analysed, "(b) all sessions passing QC": sess.QC == "passed",
            "(c) all 50 sessions": pd.Series(True, index=sess.index)}
lim = {"S": ((15, 1.5), (224, 22.4)), "D": ((159, 15.9), (1636, 163.6))}
for name, sel in readings.items():
    ok_all = True
    for plane in ("S", "D"):
        n = sess[sel & (sess.plane == plane)].n_rois
        (lo, tlo), (hi, thi) = lim[plane]
        ok = abs(n.min() - lo) <= tlo and abs(n.max() - hi) <= thi
        ok_all &= ok
        print(f"{name}, {'somata' if plane == 'S' else 'dendrites'}: {len(n)} sessions, {n.min()} to {n.max()} "
              f"(expected {lo} +/- {tlo} to {hi} +/- {thi}): {'within' if ok else 'outside'}")
    print(f"  {name}: {'all within tolerance' if ok_all else 'not all within tolerance'}")

**Result: reproduced.** In the 33 analysed sessions somatic planes hold 15 to 224 ROIs and dendritic
planes 159 to 1636, as stated. The readings over all sessions that passed QC (15 to 224, 159 to
1636) and over all 50 sessions (15 to 227, 159 to 1636) are also within tolerance.

## tracked-roi-totals

> Matching produced 254 tracked ROIs in L2/3 dendritic planes, 261 in L2/3 somatic, 516 in L5
> dendritic and 129 in L5 somatic, about 100 to 500 per plane and about 32 to 75% of the
> theoretical maximum number of trackable ROIs.

In [ ]:
trk = sess[sess.n_tracked > 0]
per_mouse = trk.groupby("subject").agg(group=("group", "first"), tracked=("n_tracked", "first"),
                                       tracked_sessions=("n_tracked", "size"), equal=("n_tracked", lambda v: v.nunique() == 1),
                                       min_rois=("n_rois", "min"), mean_rois=("n_rois", "mean"),
                                       local=("local_n", lambda v: sorted(v)))
print(per_mouse.to_string())
expected = {"L2/3-D": 254, "L2/3-S": 261, "L5-D": 516, "L5-S": 129}
for label, sel in (("11 mice of the paper's analyses", ~per_mouse.index.isin(["411400", "412933"])),
                   ("all 13 tracked mice (variant)", per_mouse.index == per_mouse.index)):
    pm = per_mouse[sel]
    tot = pm.groupby("group").agg(tracked=("tracked", "sum"), max_a=("min_rois", "sum"), max_b=("mean_rois", "sum"))
    tot["expected"] = pd.Series(expected)
    tot["pct_a"] = 100 * tot.tracked / tot.max_a
    tot["pct_b"] = 100 * tot.tracked / tot.max_b
    print(f"\n{label}:")
    print(tot.round(1).to_string())
    print("  totals equal the expected:", bool((tot.tracked == tot.expected).all()))
    for r in ("a", "b"):
        lo, hi = tot[f"pct_{r}"].min(), tot[f"pct_{r}"].max()
        print(f"  reading ({r}): lowest {lo:.1f}% (32 +/- 3.2), highest {hi:.1f}% (75 +/- 7.5):",
              "within" if abs(lo - 32) <= 3.2 and abs(hi - 75) <= 7.5 else "outside")

**Result: reproduced.** Over the 11 mice of the paper's analyses, the tracked ROIs sum to 254 (L2/3
dendrites), 261 (L2/3 somata), 516 (L5 dendrites) and 129 (L5 somata), as stated. With the
theoretical maximum read as (a), the smallest ROI count of each mouse's three tracked sessions,
the percentages run from 32.9% to 75.0%, within 32 +/- 3.2 and 75 +/- 7.5. Reading (b), the mean
count, gives 25.4% to 65.7%, outside. Over all 13 tracked mice (variant) the dendritic totals
are 352 and 678, which do not match.

## tracked-sessions-overlap

> The same imaging planes were recorded over three days and many ROIs could be identified in
> each day's recording, so that individual dendritic segments and somata can be followed over
> the three sessions.

The criterion is on the tracking indices. The mask centroids of tracked ROIs are added as a check
that an index names the same place in the field of view in each session (not part of the
criterion; the masks are not registered, so small shifts between sessions remain).

In [ ]:
ov = []
for subj, g in trk.groupby("subject"):
    ids = [np.sort(S[sid]["tracking_id"][np.isfinite(S[sid]["tracking_id"])]) for sid in g.index]
    one_each = all(len(np.unique(i)) == len(i) for i in ids)
    same_set = all(np.array_equal(ids[0], i) for i in ids)
    c = [S[sid]["tracked_centroids"] for sid in g.index]
    d_same, d_other = [], []
    for a in range(len(c)):
        for b in range(a + 1, len(c)):
            ca, cb = c[a].to_numpy(), c[b].reindex(c[a].index).to_numpy()
            d_same.append(np.hypot(*(ca - cb).T))
            dist = np.hypot(ca[:, None, 0] - cb[None, :, 0], ca[:, None, 1] - cb[None, :, 1])
            np.fill_diagonal(dist, np.inf)
            d_other.append(dist.min(axis=1))
    d_same, d_other = np.concatenate(d_same), np.concatenate(d_other)
    ov.append({"subject": subj, "group": g.group.iloc[0], "sessions": len(g), "tracked": len(ids[0]),
               "equal_counts": len({len(i) for i in ids}) == 1, "each_index_once": one_each, "same_indices": same_set,
               "median_px_same_index": np.median(d_same), "median_px_nearest_other": np.median(d_other),
               "frac_same_closer": np.mean(d_same < d_other)})
ov = pd.DataFrame(ov).set_index("subject")
print(ov.round(2).to_string())
print("Tracked mice:", len(ov), "; with three sessions, equal counts, one-to-one indices:",
      int((ov.sessions.eq(3) & ov.equal_counts & ov.each_index_once & ov.same_indices).sum()))

**Result: reproduced.** All 13 tracked mice have three tracked sessions with equal numbers of
tracked ROIs (12 to 353). In every one, each tracking index appears once per session and the
three sessions carry the same indices. In the added mask check, the same index lies a median of
1.9 to 14.5 pixels apart between sessions, against 11.5 to 62.4 pixels to the nearest other
tracked ROI. The same index is closer than any other tracked ROI for 55% (411400) to 100% of
ROIs. The masks are not registered between sessions, so this check is consistent with the
tracking, not a proof of it.

## fig8a-final-matches and fig8b-final-matches

> In the example L2/3 dendritic mouse of Fig. 8a, removing four conflicting pairwise matches
> from the union over session orderings left 136 final matches.

> In the example L5 somatic mouse of Fig. 8b, removing two conflicting pairwise matches left 47
> final matches.

In [ ]:
for subj, exp in (("413663", 136), ("411771", 47)):
    print(subj, sess.group[sess.subject == subj].iloc[0], "tracked ROIs per session:",
          sess[sess.subject == subj].n_tracked.tolist(), "; expected", exp)

**Result: reproduced (both).** Mouse 413663 has 136 tracked ROIs in each of its three sessions, and
mouse 411771 has 47.

## fig8a-conflicts, fig8b-conflicts and fig8-somata-less-order-dependent

> For the same mouse, the union of matches over all session orderings held four conflicting
> matches, which were removed.

> For the same L5 somatic mouse, two conflicting matches were removed from the union over
> session orderings.

> The number of matched ROIs varied less across session orderings for somata than for
> dendrites.

The union counts and the fewest and most matches over orderings come from the paper's
roi_tracking/nway-matched-dfs files (commit in the helper); the tracked counts from the dataset.

In [ ]:
ex_rows = []
for subj in ("413663", "411771"):
    for sid in sess.index[sess.subject == subj]:
        ex = oc.nway_example(subj, sid)["match_perm_examples"]
        lv = {ex["match_level"][k]: ex["n_total"][k] for k in ex["match_level"]}
        ex_rows.append({"subject": subj, "session": sid, "group": sess.group[sid], "union": lv["union"],
                        "fewest": lv["fewest"], "most": lv["most"], "tracked_in_dataset": sess.n_tracked[sid]})
ex_df = pd.DataFrame(ex_rows)
ex_df["conflicts"] = ex_df.union - ex_df.tracked_in_dataset
ex_df["range"] = ex_df.most - ex_df.fewest
ex_df["range_frac"] = ex_df.range / ex_df.tracked_in_dataset
print(ex_df.round(4).to_string(index=False))
listing = oc.github_listing(oc.OSCA_REPO, oc.OSCA_COMMIT, "roi_tracking/nway-matched-dfs")
with_perm = sorted({name.split("__")[0].split("_")[1] for name in listing
                    if "match_perm_examples" in oc.nway_example(*[p.split("_")[1] for p in name.split("__")[:2]])})
print("nway-matched-dfs files:", len(listing), "for", len({n.split("__")[0] for n in listing}),
      "mice; mice whose files hold per-ordering counts (match_perm_examples):", with_perm)

**Result: fig8a-conflicts and fig8b-conflicts reproduced; fig8-somata-less-order-dependent
blocked.** The repository gives union counts of 140 (413663) and 49 (411771), so 140 - 136 = 4
and 49 - 47 = 2 conflicts, as stated. The repository's files hold per-ordering counts for these
two mice only, so the comparison across mice cannot be made. In the two examples the range over
orderings is 3 matches for the dendritic mouse (129 to 132, 2.2% of its final count) and 1 for the
somatic mouse (47 to 48, 2.1%). That is smaller for somata in absolute terms but about equal as a
fraction.

## gabor-sequence-structure

> Gabor sequences are repeating 1.5-s sequences of five 300-ms images (A-B-C-D-G), each non-grey
> image holding 30 Gabor patches, with inconsistent sequences replacing D by U, whose mean
> orientation is shifted by 90 degrees (drawn from 90, 135, 180 and 225 degrees, against 0, 45,
> 90 and 135 degrees for the other images).

In [ ]:
gs = []
for sid in sess.index:
    g = S[sid]["trials"]
    g = g[g.stimulus_type == "gabors"].reset_index(drop=True)
    frames = g.gabor_frame.to_numpy()
    seqs = frames.reshape(-1, 5) if len(frames) % 5 == 0 else None
    order_ok = seqs is not None and all(tuple(r) in {("A", "B", "C", "D", "G"), ("A", "B", "C", "U", "G")} for r in seqs)
    dur = (g.stop_time - g.start_time).to_numpy()
    ori = g.gabor_mean_orientation.to_numpy().reshape(-1, 5)
    is_u = seqs[:, 3] == "U"
    abc_same = np.all(ori[:, :4][:, :3] == ori[:, [0]], axis=1)
    gs.append({"session": sid, "images": len(g), "sequences": len(seqs), "order_ok": order_ok,
               "dur_min": dur.min(), "dur_median": np.median(dur), "dur_max": dur.max(),
               "dur_within_10pct": np.mean((dur >= 0.27) & (dur <= 0.33)),
               "seq_dur_median": np.median(g.start_time.to_numpy()[5::5] - g.start_time.to_numpy()[:-5:5]),
               "patches": sorted(set(g.n_patches[g.gabor_frame != "G"])),
               "gabor_number": sorted(set(g.gabor_number.dropna())),
               "ABCD_oris": sorted(set(ori[:, :3].ravel()) | set(ori[~is_u, 3])),
               "U_oris": sorted(set(ori[is_u, 3])),
               "U_is_seq_plus_90": bool(np.all(ori[is_u, 3] == ori[is_u, 0] + 90)),
               "ABC_share_mean": bool(abc_same.all()), "kappa": sorted(set(g.gabor_kappa))})
gs = pd.DataFrame(gs).set_index("session")
print(gs.drop(columns=["patches", "gabor_number", "ABCD_oris", "U_oris", "kappa"]).round(4).to_string())
print("\nPatches per non-grey image:", {str(x) for x in gs.patches}, "; gabor_number:", {str(x) for x in gs.gabor_number})
print("A to D mean orientations:", {str(x) for x in gs.ABCD_oris}, "; U:", {str(x) for x in gs.U_oris}, "; kappa:", {str(x) for x in gs.kappa})
gs["images_outside"] = [int((~(S[sid]["trials"].query("stimulus_type == 'gabors'").eval("stop_time - start_time").between(0.27, 0.33))).sum()) for sid in gs.index]
print("Images outside 270 to 330 ms, per session where any:", gs.images_outside[gs.images_outside > 0].to_dict(),
      "; total", gs.images_outside.sum(), "of", gs.images.sum())
print("Sessions with A-B-C-D/U-G order:", int(gs.order_ok.sum()), "; with every image within 300 ms +/- 10%:",
      int((gs.dur_within_10pct == 1).sum()), "; image durations", gs.dur_min.min().round(4), "to", gs.dur_max.max().round(4), "s")
print("Sessions where U equals the sequence's mean + 90:", int(gs.U_is_seq_plus_90.sum()), "of 50")

**Result: partially reproduced.** In all 50 sessions the 1360 sequences follow A-B-C-D-G or
A-B-C-U-G, every non-grey image holds 30 Gabor patches, A to D take mean orientations 0, 45, 90
and 135 degrees, and U takes 90, 135, 180 and 225 degrees, always the sequence's mean plus 90.
Image durations have a median of 300 ms. Every image lies within 300 ms +/- 10% in 46 of 50
sessions. Four sessions hold 1 to 5 images outside it (up to 604 ms), 13 of 340000 images in all; the
three sessions of mouse 411400 among them failed QC, and Supplementary Table 1 notes dropped
stimulus frames for them. The criterion asks for every
image in every session, so the duration part holds in 46 of 50 sessions.

## visual-flow-structure

> Visual flow is 105 white squares of 8 by 8 degrees moving at 50 degrees per second, in two
> consecutive periods (rightward and leftward) in random order, with 25% of squares reversed for
> 2 to 4 s during inconsistent flow.

In [ ]:
vf = []
for sid in sess.index:
    t = S[sid]["trials"].reset_index(drop=True)
    v = t[t.stimulus_type == "visflow"]
    dirs = v.main_flow_direction.to_numpy()
    changes = int((dirs[1:] != dirs[:-1]).sum())
    contiguous = all(np.all(np.diff(v.index[v.main_flow_direction == d]) == 1) for d in set(dirs))
    runs = []
    for d in set(dirs):
        u = v[v.main_flow_direction == d].unexpected.to_numpy()
        dd = v[v.main_flow_direction == d]
        edges = np.flatnonzero(np.diff(np.concatenate([[0], u, [0]])))
        for a, b in zip(edges[::2], edges[1::2]):
            runs.append(dd.stop_time.iloc[b - 1] - dd.start_time.iloc[a])
    sq = S[sid]["squares"]
    speed = {}
    for k, (x, y, d) in sq.items():
        dx = np.diff(x.astype(float), axis=1)
        step = np.median(np.abs(dx[np.abs(dx) < 100]))           # px per stimulus frame (60 Hz), wraps excluded
        speed[k] = step * 60
        speed[k + "_dir"] = d
        # Squares moving against the main flow at the trial's first frame (reversed squares that
        # leave the screen re-enter with the main flow, so the fraction falls later in the trial).
        moving = np.where(np.abs(dx[:, 0]) < 100, np.sign(dx[:, 0]), 0)
        speed[k + "_frac_against"] = np.mean(moving == -np.sign(np.median(dx)))
    vf.append({"session": sid, "squares": sorted(set(v.square_number)), "size_px": sorted(set(v.square_size)),
               "flipped": sorted(set(v.square_proportion_flipped)), "direction_changes": changes,
               "periods_contiguous": contiguous, "first_direction": dirs[0],
               "reversal_min_s": min(runs), "reversal_max_s": max(runs), "reversals": len(runs),
               "speed_px_s": speed["consistent"], "frac_squares_against_in_reversal": speed["inconsistent_frac_against"]})
vf = pd.DataFrame(vf).set_index("session")
print(vf.drop(columns=["squares", "size_px", "flipped"]).round(3).to_string())
print("\nSquares:", {str(x) for x in vf.squares}, "; size (px):", {str(x) for x in vf.size_px},
      "; proportion reversed:", {str(x) for x in vf.flipped})
print("Sessions with two consecutive direction periods:", int(((vf.direction_changes == 1) & vf.periods_contiguous).sum()),
      "; first direction:", vf.first_direction.value_counts().to_dict())
print("Reversal durations (s):", vf.reversal_min_s.min().round(3), "to", vf.reversal_max_s.max().round(3))
print("Square speed (px/s):", vf.speed_px_s.min(), "to", vf.speed_px_s.max(), "; speed / size (1/s):",
      (vf.speed_px_s / 128).min(), "to", (vf.speed_px_s / 128).max(), "; paper 50 deg/s over 8 deg =", 50 / 8)
print("Fraction of squares moving against the flow at the first frame of the first reversed trial:",
      vf.frac_squares_against_in_reversal.min().round(3), "to", vf.frac_squares_against_in_reversal.max().round(3),
      "; median", vf.frac_squares_against_in_reversal.median().round(3), "; mean", vf.frac_squares_against_in_reversal.mean().round(3))
print("Not recorded: square size and speed in degrees (the table gives pixels of the unwarped stimulus).")

**Result: reproduced.** In all 50 sessions the table records 105 squares and a reversed proportion
of 0.25, and the two directions run as two consecutive periods (rightward first in 26 sessions).
Reversals last 1.99 to 4.01 s. From the square positions, a median of 24.8% of squares (15.2 to
32.4% by session) move against the flow at the first frame of a session's first reversal. The size (8 degrees) and speed (50 degrees/s) cannot be tested: the
table gives the unwarped stimulus in pixels, 128 pixels and 780 pixels/s. Their ratio, 6.09 per
second, is within 3% of the paper's 50/8 = 6.25.

## session-stimulus-durations

> Each imaging session lasted 70 min, with 34 min of Gabor sequences and 17 min of visual flow in
> each direction, each stimulus period flanked by 30 s of grey screen.

In [ ]:
sd = []
for sid in sess.index:
    t = S[sid]["trials"]
    per = t.groupby("stimulus_template_name").agg(start=("start_time", "min"), stop=("stop_time", "max"))
    grey = t[t.stimulus_type == "grayscreen"]
    sd.append({"session": sid, "gabor_min": (per.stop - per.start)["gabors"] / 60,
               "flow_right_min": (per.stop - per.start)["visflow_right"] / 60,
               "flow_left_min": (per.stop - per.start)["visflow_left"] / 60,
               "grey_min_s": (grey.stop_time - grey.start_time).min(), "grey_max_s": (grey.stop_time - grey.start_time).max(),
               "grey_periods": len(grey)})
sd = pd.DataFrame(sd).set_index("session")
sd["ok"] = (sd.gabor_min.between(30.6, 37.4) & sd.flow_right_min.between(15.3, 18.7) & sd.flow_left_min.between(15.3, 18.7)
            & (sd.grey_min_s >= 27) & (sd.grey_max_s <= 33))
print(sd.describe().round(3).to_string())
print("Sessions meeting all three:", int(sd.ok.sum()), "of 50")

**Result: reproduced.** In all 50 sessions the Gabor period lasts 34.03 to 34.31 min, each flow
period 17.01 to 17.04 min, and each of the four grey periods 29.98 to 30.04 s.

## inconsistent-fractions

> Inconsistent sequences make up about 7% of Gabor sequences and 5% of visual flow time.

In [ ]:
fr = []
for sid in sess.index:
    t = S[sid]["trials"]
    g = t[t.stimulus_type == "gabors"]
    v = t[t.stimulus_type == "visflow"]
    vdur = v.stop_time - v.start_time
    fr.append({"session": sid, "gabor_pct": 100 * (g.gabor_frame == "U").sum() / ((g.gabor_frame == "D").sum() + (g.gabor_frame == "U").sum()),
               "flow_time_pct": 100 * vdur[v.unexpected == 1].sum() / vdur.sum()})
fr = pd.DataFrame(fr).set_index("session")
print(fr.round(2).T.to_string())
print("Mean over sessions: Gabor", fr.gabor_pct.mean().round(2), "% (7 +/- 0.7); flow", fr.flow_time_pct.mean().round(2),
      "% (5 +/- 0.5); ranges", fr.gabor_pct.min().round(2), "to", fr.gabor_pct.max().round(2), "and",
      fr.flow_time_pct.min().round(2), "to", fr.flow_time_pct.max().round(2))

**Result: reproduced.** Over the 50 sessions, inconsistent sequences make up 6.96% of Gabor
sequences (6.03 to 7.94% per session; 7 +/- 0.7) and reversed flow 4.68% of flow time (3.92 to
5.49%; 5 +/- 0.5).

## inconsistent-blocks

> Each session is divided into about 30 blocks, each of 30 to 90 s of consistent stimulus
> followed by 3 to 6 s (Gabor) or 2 to 4 s (visual flow) of inconsistent stimulus, in multiples
> of 1.5 s (Gabor) and 1 s (visual flow).

A block is a run of consistent stimulus followed by a run of inconsistent stimulus, within one
stimulus period. Gabor runs are counted in sequences (the unexpected flag is set for the whole
A-B-C-U-G sequence), flow runs in 1-s trials.

In [ ]:
blocks = []
for sid in sess.index:
    t = S[sid]["trials"]
    for name in ("gabors", "visflow_right", "visflow_left"):
        x = t[t.stimulus_template_name == name]
        if name == "gabors":
            x = x[x.gabor_frame == "A"]                          # one row per sequence
            ends = t[(t.stimulus_template_name == name) & (t.gabor_frame == "G")].stop_time.to_numpy()
        else:
            ends = x.stop_time.to_numpy()
        u, st = x.unexpected.to_numpy(), x.start_time.to_numpy()
        i = 0
        while i < len(u):
            j = i
            while j < len(u) and u[j] == 0:
                j += 1
            k = j
            while k < len(u) and u[k] == 1:
                k += 1
            if k > j:   # a consistent run followed by an inconsistent run
                blocks.append({"session": sid, "stimulus": name, "consistent_s": st[j] - st[i], "inconsistent_s": ends[k - 1] - st[j],
                               "n_units_inconsistent": k - j, "n_units_consistent": j - i})
            i = k if k > i else i + 1
blocks = pd.DataFrame(blocks)
blocks["kind"] = np.where(blocks.stimulus == "gabors", "gabor", "flow")
inc_lim = {"gabor": (3 * 0.9, 6 * 1.1), "flow": (2 * 0.9, 4 * 1.1)}
blocks["consistent_ok"] = blocks.consistent_s.between(30 * 0.9, 90 * 1.1)
blocks["inconsistent_ok"] = [inc_lim[k][0] <= v <= inc_lim[k][1] for k, v in zip(blocks.kind, blocks.inconsistent_s)]
print(blocks.groupby("stimulus")[["consistent_s", "inconsistent_s"]].describe().round(2).T.to_string())
print("\nInconsistent runs, in units (Gabor sequences of 1.5 s, flow trials of 1 s):",
      blocks.groupby("kind").n_units_inconsistent.value_counts().sort_index().to_dict())
counts = {"(a) all stimuli": blocks.groupby("session").size(),
          "(b) Gabor": blocks[blocks.kind == "gabor"].groupby("session").size(),
          "(b) flow, both directions": blocks[blocks.kind == "flow"].groupby("session").size(),
          "(c) flow right": blocks[blocks.stimulus == "visflow_right"].groupby("session").size(),
          "(c) flow left": blocks[blocks.stimulus == "visflow_left"].groupby("session").size()}
for k, v in counts.items():
    print(f"{k}: mean blocks per session {v.mean():.2f} (range {v.min()} to {v.max()}); within 30 +/- 3: {abs(v.mean() - 30) <= 3}")
for kind in ("gabor", "flow"):
    b = blocks[blocks.kind == kind]
    print(f"{kind}: blocks with consistent and inconsistent durations in range (widened 10%): "
          f"{100 * (b.consistent_ok & b.inconsistent_ok).mean():.1f}% (consistent {100 * b.consistent_ok.mean():.1f}%, "
          f"inconsistent {100 * b.inconsistent_ok.mean():.1f}%)")
print("All blocks:", f"{100 * (blocks.consistent_ok & blocks.inconsistent_ok).mean():.1f}%")

**Result: reproduced.** Under reading (b), per stimulus, the sessions hold a mean of 31.5 Gabor
blocks and 32.3 flow blocks (both directions together), both within 30 +/- 3. Reading (a), all
stimuli together, gives 63.8, and reading (c), per flow direction, gives 16.0 and 16.3; both are
outside. Every block has 30.0 to 91.0 s of consistent stimulus and 3.0 to 6.1 s (Gabor; 2 to 4
sequences of 1.5 s) or 2.0 to 4.0 s (flow; 2 to 4 one-second trials) of inconsistent stimulus.

## fig5-example-responses

> Both somatic and dendritic ROIs responded clearly to the Gabor and visual flow stimuli, many
> with increased fluorescence at stimulus onset.

Session 1 of the 11 mice of the paper's analyses. The selection is made within each mouse's
session (helper `select_gabor_rois` and `select_flow_rois`), on robust-scaled dF/F; the
selected ROIs are pooled per line and plane. Panels: (a) consistent sequences around D onset,
(b) inconsistent sequences around U onset, (c) temporal-to-nasal (leftward) flow and (d)
nasal-to-temporal (rightward) flow, around inconsistent flow onset.

In [ ]:
s1 = sess[sess.analysed & (sess.local_n == 1)]
panels = {"a": ("gabor_D", "gabor_t"), "b": ("gabor_U", "gabor_t"), "c": ("flow_left", "flow_t"), "d": ("flow_right", "flow_t")}
groups = ["L2/3-D", "L2/3-S", "L5-D", "L5-S"]


def pooled(group, key, scaled="scaled", sel_key=None):
    """Selected ROIs' mean traces, pooled over the group's session-1 recordings, with the mouse of each."""
    traces, mouse = [], []
    for sid in s1.index[s1.group == group]:
        f5 = R[sid]["fig5"]
        sel = f5[f"{sel_key or key}_{scaled}_sel"]
        traces.append(f5[f"{key}_{scaled}_mean"][sel])
        mouse += [sess.subject[sid]] * len(sel)
    return np.concatenate(traces), np.array(mouse)


rows5, per_mouse5 = [], []
for scaled in ("scaled", "unscaled"):
    for p, (key, tkey) in panels.items():
        for grp in groups:
            tr, mouse = pooled(grp, key, scaled)
            t = R[s1.index[0]]["fig5"][tkey]
            pre = tr[:, t < 0].mean() if len(tr) else np.nan
            post = tr[:, t >= 0].mean() if len(tr) else np.nan
            rows5.append({"traces": scaled, "panel": p, "group": grp, "rois": len(tr), "mice": len(set(mouse)),
                          "pre": pre, "post": post, "post_minus_pre": post - pre, "higher_after": post > pre})
            for m in sorted(set(s1.subject[s1.group == grp])):
                sel = mouse == m
                per_mouse5.append({"traces": scaled, "panel": p, "group": grp, "mouse": m, "rois": int(sel.sum()),
                                   "post_minus_pre": (tr[sel][:, t >= 0].mean() - tr[sel][:, t < 0].mean()) if sel.any() else np.nan})
rows5 = pd.DataFrame(rows5)
print("Selected ROIs, robust-scaled dF/F (main):")
print(rows5[rows5.traces == "scaled"].round(4).to_string(index=False))
print("\nPer mouse (robust-scaled):")
print(pd.DataFrame(per_mouse5).query("traces == 'scaled'").pivot_table(index=["group", "mouse"], columns="panel",
      values=["rois", "post_minus_pre"]).round(3).to_string())
n_sess1 = {sid: (R[sid]["n_rois"], int(R[sid]["fig5"]["snr_above"].sum()), R[sid]["fig5"]["gabor_D_n"], R[sid]["fig5"]["gabor_U_n"],
                 R[sid]["fig5"]["flow_left_n"], R[sid]["fig5"]["flow_right_n"]) for sid in s1.index}
print("\nPer session 1: ROIs, ROIs above median SNR, D sequences, U sequences, left and right flow onsets:", n_sess1)
for scaled in ("scaled", "unscaled"):
    r = rows5[rows5.traces == scaled]
    for p in panels:
        q = r[r.panel == p]
        print(f"{scaled}, panel {p}: planes with selected ROIs {int((q.rois > 0).sum())} of 4; higher after onset {int(q.higher_after.sum())} of 4")
fig, axes = plt.subplots(4, 4, figsize=(13, 10), sharex="col")
for j, (p, (key, tkey)) in enumerate(panels.items()):
    for i, grp in enumerate(groups):
        tr, _ = pooled(grp, key)
        t = R[s1.index[0]]["fig5"][tkey]
        ax = axes[i, j]
        if len(tr):
            m, se = tr.mean(axis=0), tr.std(axis=0, ddof=1) / np.sqrt(len(tr)) if len(tr) > 1 else 0 * tr[0]
            ax.plot(t, m, color="k")
            ax.fill_between(t, m - se, m + se, color="0.7")
        ax.axvline(0, ls="--", color="0.5")
        ax.set_title(f"{grp}, panel {p}, n = {len(tr)}", fontsize=9)
for ax in axes[-1]:
    ax.set_xlabel("time from D, U or inconsistent flow onset (s)")
fig.supylabel("robust-scaled dF/F, mean +/- SEM over selected ROIs")
fig.tight_layout()
plt.show()
lo_hi = [(pooled(g, k)[0].mean(axis=0).min(), pooled(g, k)[0].mean(axis=0).max()) for k, _ in panels.values() for g in groups if len(pooled(g, k)[0])]
print("Range of the plotted means:", round(min(a for a, _ in lo_hi), 3), "to", round(max(b for _, b in lo_hi), 3))

**Result: blocked (methods_underspecified).** The selection yields ROIs in every plane for each
panel. The flow responses (panels c and d) are higher after inconsistent flow onset in all four
planes. For Gabor sequences, panel a (consistent, around D) is higher after onset in 2 of 4
planes; L2/3 and L5 somata are lower (-0.063 and -0.020). Panel b (inconsistent, around U) is
higher in 3 of 4; L5 dendrites are at -0.001. The criterion's per-plane rule does not say which
Gabor panel it uses. With panel a and the flow panels, the rule holds in 2 planes (partially
reproduced); with panel b, in 3 planes (partially reproduced); with all four panels, in 1 plane
(not reproduced). The readings disagree, so the claim is blocked and a question was asked. With
unscaled dF/F, panel a is higher in 2 of 4 planes and panel b in 4 of 4.

## fig5-consistent-vs-inconsistent

> Responses to consistent and inconsistent stimuli differed clearly in the example ROIs.

Gabor: for the ROIs selected on consistent sequences (reading a) or on inconsistent sequences
(reading b), the mean over ROIs of their mean trace after D onset and after U onset (0 to 0.6
s), with the SEM across ROIs. Clear when the two differ by more than the sum of their SEMs at
some time point.

In [ ]:
cvi = []
t = R[s1.index[0]]["fig5"]["gabor_t"]
post = t >= 0
for scaled in ("scaled", "unscaled"):
    for reading, sel_key in (("a", "gabor_D"), ("b", "gabor_U")):
        for grp in groups:
            d, _ = pooled(grp, "gabor_D", scaled, sel_key)
            u, _ = pooled(grp, "gabor_U", scaled, sel_key)
            n = len(d)
            if n < 2:
                cvi.append({"traces": scaled, "reading": reading, "group": grp, "rois": n, "clear": False})
                continue
            md, mu = d.mean(axis=0)[post], u.mean(axis=0)[post]
            sd, su = d.std(axis=0, ddof=1)[post] / np.sqrt(n), u.std(axis=0, ddof=1)[post] / np.sqrt(n)
            gap = np.abs(mu - md) - (sd + su)
            cvi.append({"traces": scaled, "reading": reading, "group": grp, "rois": n,
                        "max_abs_diff": np.abs(mu - md).max(), "sem_sum_there": (sd + su)[np.argmax(gap)],
                        "frames_clear": int((gap > 0).sum()), "frames": int(post.sum()), "clear": bool((gap > 0).any()),
                        "mean_U_minus_D": (mu - md).mean()})
cvi = pd.DataFrame(cvi)
print(cvi.round(4).to_string(index=False))
for scaled in ("scaled", "unscaled"):
    for reading in ("a", "b"):
        q = cvi[(cvi.traces == scaled) & (cvi.reading == reading)]
        print(f"{scaled}, reading ({reading}): planes with a clear difference {int(q.clear.sum())} of 4")
# Visual flow: reported, not judged (the selection requires a larger response after onset).
for p in ("c", "d"):
    q = rows5[(rows5.traces == "scaled") & (rows5.panel == p)]
    print(f"flow panel {p}: post minus pre over selected ROIs", dict(zip(q.group, q.post_minus_pre.round(3))))

**Result: blocked (methods_underspecified), as it depends on fig5-example-responses.** Its own
test ran: with robust-scaled traces, the D and U responses of the selected ROIs differ by more than
the sum of their SEMs in 2 of 4 planes under reading (a), the ROIs selected on consistent
sequences, and in 3 of 4 under reading (b), the ROIs selected on inconsistent sequences. By its
criterion this is partially reproduced under both readings. The flow differences (post minus pre,
0.030 to 0.206) are reported only, as the selection requires them.

## fig6-behaviour-all-mice

> Running velocity and pupil diameter were recorded in sessions 1 to 3 of all 11 mice, so their
> distributions can be shown per session across mice.

In [ ]:
b6 = pd.DataFrame({sid: {"subject": sess.subject[sid], "local_n": sess.local_n[sid],
                         "run_unit": S[sid]["run_unit"], "run_finite": np.isfinite(S[sid]["run_v"]).mean(),
                         "pupil_unit": S[sid]["pupil_unit"], "pupil_finite": np.isfinite(S[sid]["pupil_d"]).mean()}
                   for sid in sess.index[sess.analysed]}).T
b6["counts"] = (b6.run_unit == "cm/s") & (b6.pupil_unit == "mm") & (b6.run_finite > 0) & (b6.pupil_finite > 0)
print(b6.to_string())
print("Sessions counting:", int(b6.counts.sum()), "of", len(b6), "; mice:", b6[b6.counts].subject.nunique(),
      "; fewest finite fractions: running", b6.run_finite.min(), ", pupil", round(b6.pupil_finite.min(), 4))

**Result: reproduced.** All 33 analysed sessions, from 11 mice, hold running velocity in cm/s
and pupil diameter in mm. At least 99.99% of running samples and 97.7% of pupil samples are
finite.

## fig7a-snr-above-1

> ROI dF/F traces have a relatively high SNR, above 1.

SNR per ROI on its full-session unscaled dF/F (helper `calculate_snr`, a port of the paper's
code). The 33 analysed sessions; ROIs pooled over mice per line, plane and session (12 groups).
Readings: (a) each group's median above 1, (b) each group's 5th percentile above 1, (c) every
ROI above 1.

In [ ]:
roi_all = pd.concat([R[sid]["roi"].assign(session=sid, subject=sess.subject[sid], group=sess.group[sid],
                                          local_n=sess.local_n[sid]) for sid in sess.index[sess.analysed]], ignore_index=True)
g7 = roi_all.groupby(["group", "local_n"])
snr_tab = g7.snr.agg(rois="size", mice=lambda v: 0, p5=lambda v: np.percentile(v, 5), q25=lambda v: np.percentile(v, 25),
                     median="median", q75=lambda v: np.percentile(v, 75), p95=lambda v: np.percentile(v, 95), min="min", max="max")
snr_tab["mice"] = g7.subject.nunique()
print(snr_tab.round(3).to_string())
print("(a) groups with median above 1:", int((snr_tab["median"] > 1).sum()), "of 12")
print("(b) groups with 5th percentile above 1:", int((snr_tab.p5 > 1).sum()), "of 12")
print("(c) ROIs above 1:", int((roi_all.snr > 1).sum()), "of", len(roi_all), "; lowest SNR", roi_all.snr.min().round(3))
print("ROIs with median dF/F below 0 (the only case where SNR can fall below 1.645):", int((roi_all["median"] < 0).sum()))
print("\nPer mouse, lowest and median SNR:")
print(roi_all.groupby(["group", "subject"]).snr.agg(["size", "min", "median"]).round(3).to_string())

**Result: reproduced.** Under all three readings: every one of the 12 groups has a median SNR above 1
(2.37 to 3.43) and a 5th percentile above 1 (2.27 to 2.41), and all 10736 ROIs exceed 1 (lowest
2.16). No ROI has a negative median dF/F, so by the SNR's construction none could fall below
1.645.

## fig7a-snr-stable and fig7b-signal-stable

> ROI SNR is stable over the three days.

> The mean dF/F signal of ROIs is stable over days.

Signal is the mean of the dF/F values above the noise threshold (`calculate_snr`). Readings:
(a) the notches (median +/- 1.57 IQR / sqrt(n)) of the three sessions of a plane overlap
pairwise; (b) each session's median lies inside the interquartile range of every other session
of the plane.

In [ ]:
def stability(col):
    rows = []
    for grp in groups:
        st = {}
        for n in (1, 2, 3):
            v = roi_all[(roi_all.group == grp) & (roi_all.local_n == n)][col].to_numpy()
            q1, med, q3 = np.percentile(v, [25, 50, 75])
            half = 1.57 * (q3 - q1) / np.sqrt(len(v))
            st[n] = (med, q1, q3, med - half, med + half, len(v))
        notch = all(st[a][3] <= st[b][4] and st[b][3] <= st[a][4] for a, b in ((1, 2), (1, 3), (2, 3)))
        iqr = all(st[b][1] <= st[a][0] <= st[b][2] for a in (1, 2, 3) for b in (1, 2, 3) if a != b)
        rows.append({"group": grp, **{f"median_s{n}": st[n][0] for n in (1, 2, 3)},
                     **{f"notch_s{n}": f"{st[n][3]:.3f}-{st[n][4]:.3f}" for n in (1, 2, 3)},
                     **{f"iqr_s{n}": f"{st[n][1]:.3f}-{st[n][2]:.3f}" for n in (1, 2, 3)},
                     "notches_overlap": notch, "medians_in_iqrs": iqr})
    return pd.DataFrame(rows).set_index("group")


for col, name in (("snr", "SNR"), ("signal", "signal (dF/F)")):
    stab = stability(col)
    print(f"{name}:")
    print(stab.round(3).to_string())
    print(f"  planes stable under (a): {int(stab.notches_overlap.sum())} of 4; under (b): {int(stab.medians_in_iqrs.sum())} of 4;"
          f" under both: {int((stab.notches_overlap & stab.medians_in_iqrs).sum())} of 4\n")
print("Signal range (dF/F) over ROIs:", roi_all.signal.min().round(3), "to", roi_all.signal.max().round(3))
print("\nPer mouse, median SNR and signal by session:")
print(roi_all.pivot_table(index=["group", "subject"], columns="local_n", values=["snr", "signal"], aggfunc="median").round(3).to_string())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ("snr", "signal")):
    data = [roi_all[(roi_all.group == g) & (roi_all.local_n == n)][col] for g in groups for n in (1, 2, 3)]
    ax.boxplot(data, notch=True, whis=(5, 95), showfliers=False)
    ax.set_xticks(range(1, 13), [f"{g} s{n}" for g in groups for n in (1, 2, 3)], rotation=90, fontsize=8)
    ax.set_ylabel(col)
    if col == "snr":
        ax.axhline(1, ls="--", color="0.5")
fig.tight_layout()
plt.show()
print("Outliers not drawn (beyond the 5th to 95th percentile whiskers).")

**Result: both blocked (methods_underspecified), as their readings disagree.** For SNR, every
session's median lies inside the other sessions' interquartile ranges in all four planes (reading
b). The notches overlap pairwise in only 2 of 4 planes (reading a); in L2/3 and L5 dendrites,
whose sessions hold 648 to 2807 ROIs, the notches are narrow and the medians drift by 0.016 and
0.033. For signal, reading (b) holds in 4 of 4 planes and reading (a) in 1 of 4. Under reading
(a) both claims would be not reproduced, under (b) reproduced. The criteria name
methods_underspecified for this case.

## fig7c-correlations-similar and fig7c-high-correlation-fraction

> The distributions of pairwise ROI correlations are very similar for apical dendrite ROIs and
> somatic ROIs, which suggests the dendrites are not heavily over-segmented.

> About 0.01% of possible pairs of L2/3 dendritic ROIs have correlations above 0.8.

Pearson correlation of every pair of ROIs within a session, on full-session dF/F smoothed with
a centred 4-point moving average (mode mirror), pooled over mice per line, plane and session.
Similar: medians, and 95th percentiles, of the dendritic and somatic distributions of the same
line and session differ by less than (a) 0.05 or (b) 0.1.

In [ ]:
corr = {}
for grp in groups:
    for n in (1, 2, 3):
        sids = sess.index[sess.analysed & (sess.group == grp) & (sess.local_n == n)]
        corr[(grp, n)] = np.concatenate([R[sid]["corr"] for sid in sids])
ctab = pd.DataFrame({k: {"pairs": len(v), "median": np.median(v), "p95": np.percentile(v, 95), "min": v.min(), "max": v.max(),
                         "pct_above_0.8": 100 * np.mean(v > 0.8), "pct_at_least_0.8": 100 * np.mean(v >= 0.8)}
                     for k, v in corr.items()}).T
print(ctab.round(4).to_string())
sim = []
for line in ("L2/3", "L5"):
    for n in (1, 2, 3):
        d, s_ = ctab.loc[(f"{line}-D", n)], ctab.loc[(f"{line}-S", n)]
        sim.append({"line": line, "session": n, "median_diff": abs(d["median"] - s_["median"]), "p95_diff": abs(d.p95 - s_.p95)})
sim = pd.DataFrame(sim)
for k, thr in (("a", 0.05), ("b", 0.1)):
    sim[f"similar_{k}"] = (sim.median_diff < thr) & (sim.p95_diff < thr)
print(sim.round(4).to_string(index=False))
print("Comparisons similar under (a):", int(sim.similar_a.sum()), "of 6; under (b):", int(sim.similar_b.sum()), "of 6;"
      " under both:", int((sim.similar_a & sim.similar_b).sum()))
print("\nPer mouse and session, median and 95th percentile of correlations:")
print(pd.DataFrame({(sess.group[sid], sess.subject[sid], sess.local_n[sid]): {"pairs": len(R[sid]["corr"]), "median": np.median(R[sid]["corr"]),
                    "p95": np.percentile(R[sid]["corr"], 95)} for sid in sess.index[sess.analysed]}).T.sort_index().round(4).to_string())
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(-1, 1, 41)
for (grp, n), v in corr.items():
    if n == 1:
        ax.hist(v, bins=bins, histtype="step", density=True, label=grp, log=True)
ax.set_xlabel("pairwise correlation (session 1)")
ax.set_ylabel("density (log)")
ax.legend()
plt.show()
print("All correlations lie in [-1, 1]; bins of 0.05 as in the code.")

print("\nL2/3 dendritic pairs with r > 0.8 (expected 0.01 +/- 0.005 %):")
l23d = {n: corr[("L2/3-D", n)] for n in (1, 2, 3)}
pooled_all = np.concatenate(list(l23d.values()))
per_ms = {(sess.subject[sid], sess.local_n[sid]): R[sid]["corr"] for sid in sess.index[sess.analysed & (sess.group == "L2/3-D")]}
for op, name in ((np.greater, "r > 0.8"), (np.greater_equal, "r >= 0.8 (variant)")):
    a = 100 * np.mean(op(pooled_all, 0.8))
    b = np.mean([100 * np.mean(op(v, 0.8)) for v in l23d.values()])
    b2 = np.mean([100 * np.mean(op(v, 0.8)) for v in per_ms.values()])
    c = {n: 100 * np.mean(op(v, 0.8)) for n, v in l23d.items()}
    c2 = {k: round(100 * np.mean(op(v, 0.8)), 4) for k, v in per_ms.items()}
    print(f"  {name}: (a) pooled {a:.4f}% ({int(op(pooled_all, 0.8).sum())} of {len(pooled_all)} pairs); "
          f"(b) mean of the three sessions {b:.4f}% (mean of the {len(per_ms)} mouse sessions {b2:.4f}%); "
          f"(c) per session {({k: round(v, 4) for k, v in c.items()})}")
    print(f"     per mouse and session: {c2}")

**Result: fig7c-correlations-similar blocked (methods_underspecified), as its readings disagree;
fig7c-high-correlation-fraction blocked as it depends on it.** The dendritic and somatic medians
differ by 0.006 to 0.025 and the 95th percentiles by 0.049 to 0.089. That is similar for 6 of 6
comparisons under reading (b), 0.1, and for 1 of 6 under reading (a), 0.05. For the high
correlations, 0.0178% of L2/3 dendritic pairs exceed 0.8 pooled (134 of 752101, reading a), and
the mean of the three sessions is 0.0176% (reading b). By session they are 0.0211, 0.0276 and
0.0043% (reading c). None is within 0.005 to 0.015%, so the claim would not be reproduced if its
dependency were decided. Two of the six mouse sessions fall within the band (0.0054 and 0.0067%).
r >= 0.8 gives the same counts.

## fig7a-source-link

> The per-ROI SNRs computed from the dataset agree with those in the source data of Figure 7a in
> the paper's repository.

In [ ]:
src = oc.fig7a_source()
src["sessids"] = src.sessids.astype(int).astype(str)
link = []
for _, r in src.iterrows():
    sid = r.sessids
    mine = R[sid]["roi"].snr.to_numpy() if sid in R else np.array([])
    theirs = np.asarray(r.snrs, dtype=float)
    same_n = len(mine) == len(theirs)
    rel = np.abs(mine - theirs) / np.abs(theirs) if same_n else np.array([np.nan])
    link.append({"session": sid, "in_dataset": sid in R, "rois_source": len(theirs), "rois_dataset": len(mine),
                 "frac_within_1pct": np.mean(rel <= 0.01) if same_n else np.nan, "max_rel_diff": np.nanmax(rel)})
link = pd.DataFrame(link).set_index("session")
link["ok"] = (link.rois_source == link.rois_dataset) & (link.frac_within_1pct >= 0.95)
print(link.to_string())
print("Source sessions:", len(link), "; in the analysed set:", int(link.index.isin(sess.index[sess.analysed]).sum()),
      "; sessions meeting the rule:", int(link.ok.sum()))

**Result: reproduced.** All 33 sessions of the Fig. 7a source data are in the dataset with the same
number of ROIs. Every ROI's SNR computed here is within 1% of the source value; the largest
relative difference is 3.4e-05.

## fig9-onset-offset-transients

> Mean dF/F responses across ROIs show distinct transients aligned to stimulus onsets (grey screen
> to stimulus) and offsets (stimulus to grey screen), which validates the temporal alignment of
> stimuli and recordings.

Onsets are the ends of every grey period but the last, offsets the starts of every grey period
but the first, -2 to 2 s around each. Per ROI, the mean trace over transitions; then mean and
SEM across ROIs pooled over mice per line, plane and session. A group shows a transient when at
some time in 0 to 2 s the mean differs from its mean over -2 to 0 s by more than k SEM.
Readings: transitions (a) all, (b) Gabor only; k 1 or 2. Robust-scaled dF/F; unscaled as a variant.

In [ ]:
def transient_table(scaled):
    rows = []
    for grp in groups:
        for n in (1, 2, 3):
            sids = sess.index[sess.analysed & (sess.group == grp) & (sess.local_n == n)]
            for kind in ("onset", "offset"):
                for reading in ("a", "b"):
                    per_roi = []
                    for sid in sids:
                        tr = R[sid]["transitions"]
                        sel = (tr.kind == kind) & ((reading == "a") | (tr.stimulus == "gabors"))
                        per_roi.append(R[sid][f"trans_{scaled}"][:, sel.to_numpy()].mean(axis=1))
                    x = np.concatenate(per_roi)
                    t = R[sids[0]]["trans_t"]
                    m, se = x.mean(axis=0), x.std(axis=0, ddof=1) / np.sqrt(len(x))
                    base = m[t < 0].mean()
                    z = np.abs(m[t >= 0] - base) / se[t >= 0]
                    rows.append({"traces": scaled, "group": grp, "session": n, "kind": kind, "reading": reading,
                                 "rois": len(x), "max_dev_in_sem": z.max(), "k1": bool((z > 1).any()), "k2": bool((z > 2).any()),
                                 "peak_change": (m[t >= 0] - base)[np.argmax(z)]})
    return pd.DataFrame(rows)


for scaled in ("scaled", "unscaled"):
    tt9 = transient_table(scaled)
    wide = tt9.pivot_table(index=["group", "session"], columns=["kind", "reading"], values="max_dev_in_sem")
    print(f"{scaled} dF/F, largest deviation from baseline in 0 to 2 s, in SEMs:")
    print(wide.round(2).to_string())
    for reading in ("a", "b"):
        for k in ("k1", "k2"):
            q = tt9[tt9.reading == reading]
            both = q.pivot_table(index=["group", "session"], columns="kind", values=k).all(axis=1)
            print(f"  reading ({reading}), {k}: groups with a transient at onset and at offset {int(both.sum())} of 12")
    print()
pm9 = []
for sid in sess.index[sess.analysed]:
    tr, t = R[sid]["transitions"], R[sid]["trans_t"]
    row = {"group": sess.group[sid], "mouse": sess.subject[sid], "session": sess.local_n[sid], "rois": R[sid]["n_rois"]}
    for kind in ("onset", "offset"):
        x = R[sid]["trans_scaled"][:, (tr.kind == kind).to_numpy()].mean(axis=1)
        if len(x) > 1:
            m, se = x.mean(axis=0), x.std(axis=0, ddof=1) / np.sqrt(len(x))
            row[kind] = (np.abs(m[t >= 0] - m[t < 0].mean()) / se[t >= 0]).max()
    pm9.append(row)
pm9 = pd.DataFrame(pm9)
print("Per mouse and session, reading (a), robust-scaled: largest deviation in SEMs over that mouse's ROIs")
print(pm9.round(2).to_string(index=False))
print("Range over mouse sessions (SEMs):", round(min(pm9.onset.min(), pm9.offset.min()), 2), "to",
      round(max(pm9.onset.max(), pm9.offset.max()), 2))
print("Mouse sessions below 2 SEM at onset or offset:", pm9[(pm9.onset <= 2) | (pm9.offset <= 2)][["mouse", "session"]].values.tolist())
fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharex=True)
for j, grp in enumerate(groups):
    for i, kind in enumerate(("onset", "offset")):
        ax = axes[i, j]
        for n in (1, 2, 3):
            sids = sess.index[sess.analysed & (sess.group == grp) & (sess.local_n == n)]
            x = np.concatenate([R[sid]["trans_scaled"][:, (R[sid]["transitions"].kind == kind).to_numpy()].mean(axis=1) for sid in sids])
            t = R[sids[0]]["trans_t"]
            m, se = x.mean(axis=0), x.std(axis=0, ddof=1) / np.sqrt(len(x))
            ax.plot(t, m, label=f"session {n}")
            ax.fill_between(t, m - se, m + se, alpha=0.3)
        ax.axvline(0, ls="--", color="0.5")
        ax.set_title(f"{grp} {kind}", fontsize=9)
axes[0, 0].legend(fontsize=8)
fig.supxlabel("time from transition (s)")
fig.supylabel("robust-scaled dF/F, mean +/- SEM over ROIs")
fig.tight_layout()
plt.show()

**Result: reproduced.** In all 12 groups the mean robust-scaled response departs from its pre-transition
mean by more than 2 SEM, at both onset and offset. This holds with all transitions (2.3 to 35.3
SEM) and with Gabor transitions only, so it holds under every reading. Unscaled dF/F gives the
same, and so does every mouse session taken alone (4.1 to 30.2 SEM, reading a).

## file-sizes and download-33-files

> Each session has three file versions; basic files are about 130 MB to 1.7 GB, the stimulus
> images add about 1.5 GB, and the motion-corrected stack adds about 45 GB.

> The basic files of the sessions 1 to 3 that passed QC are 33 files totalling about 15 GB.

In [ ]:
sz = assets.dropna(subset=["kind"]).pivot_table(index="session", columns="kind", values="size", aggfunc="first")
print("Sessions with all three versions:", int(sz.notna().all(axis=1).sum()), "of", len(sz))
for unit, mb, gb in (("decimal", 1e6, 1e9), ("binary", 2**20, 2**30)):
    vals = {"smallest basic (MB)": sz.B.min() / mb, "largest basic (GB)": sz.B.max() / gb,
            "I minus B, median (GB)": (sz.I - sz.B).median() / gb, "S minus I, median (GB)": (sz.S - sz.I).median() / gb}
    exp = {"smallest basic (MB)": (130, 13), "largest basic (GB)": (1.7, 0.17), "I minus B, median (GB)": (1.5, 0.15),
           "S minus I, median (GB)": (45, 4.5)}
    within = {k: abs(v - exp[k][0]) <= exp[k][1] for k, v in vals.items()}
    print(f"{unit}:", {k: round(v, 3) for k, v in vals.items()}, "; within tolerance:", within, "; all:", all(within.values()))
print("I minus B range (GB, decimal):", round((sz.I - sz.B).min() / 1e9, 3), "to", round((sz.I - sz.B).max() / 1e9, 3),
      "; S minus I range:", round((sz.S - sz.I).min() / 1e9, 2), "to", round((sz.S - sz.I).max() / 1e9, 2))
sel33 = sz.loc[sess.index[sess.analysed]]
print("\nBasic files of local sessions 1 to 3 passing QC:", len(sel33), "; total", round(sel33.B.sum() / 1e9, 2), "GB decimal,",
      round(sel33.B.sum() / 2**30, 2), "GB binary (expected 15 +/- 1.5)")

**Result: both reproduced.** Every session has the three file versions. In decimal units the basic
files run from 136.7 MB to 1.70 GB, the stimulus images add a median of 1.49 GB and the imaging
stack a median of 45.5 GB; in binary units 130.3 MB, 1.59 GB, 1.39 GB and 42.3 GB. All are within
tolerance in both. The 33 basic files of local sessions 1 to 3 that passed QC total 14.82 GB
(13.80 GiB).

## Summary

| Claim | Outcome |
|---|---|
| table1-subjects | reproduced |
| cohorts-by-plane | reproduced |
| sex-split | reproduced |
| sessions-total | reproduced |
| table2-sessions | not reproduced (43 of 50 sessions agree in every recorded field) |
| qc-failures | blocked: metadata insufficient (no QC status in the dataset) |
| session-schedule | reproduced |
| imaging-depths | not reproduced (mouse 433414 at 60 um) |
| acquisition-rates | reproduced |
| roi-noise-rejection | reproduced |
| roi-count-ranges | reproduced |
| tracked-roi-totals | reproduced |
| tracked-sessions-overlap | reproduced |
| fig8a-final-matches | reproduced |
| fig8a-conflicts (with the paper's matching files) | reproduced |
| fig8b-final-matches | reproduced |
| fig8b-conflicts (with the paper's matching files) | reproduced |
| fig8-somata-less-order-dependent | blocked: data missing (per-ordering counts for two mice only) |
| gabor-sequence-structure | partially reproduced (13 images in four sessions outside 300 ms +/- 10%) |
| visual-flow-structure | reproduced (size and speed in degrees not testable) |
| session-stimulus-durations | reproduced |
| inconsistent-fractions | reproduced |
| inconsistent-blocks | reproduced |
| fig5-example-responses | blocked: methods underspecified (question asked) |
| fig5-consistent-vs-inconsistent | blocked: methods underspecified (depends on fig5-example-responses) |
| fig6-behaviour-all-mice | reproduced |
| fig7a-snr-above-1 | reproduced |
| fig7a-snr-stable | blocked: methods underspecified (readings disagree) |
| fig7b-signal-stable | blocked: methods underspecified (readings disagree) |
| fig7c-correlations-similar | blocked: methods underspecified (readings disagree) |
| fig7c-high-correlation-fraction | blocked: methods underspecified (depends on fig7c-correlations-similar) |
| fig7a-source-link (with the paper's Fig. 7a source data) | reproduced |
| fig9-onset-offset-transients | reproduced |
| file-sizes | reproduced |
| download-33-files | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The 33 analysed sessions are selected with the
QC results and local session numbers of Supplementary Table 1, as the dataset holds no QC; the
local numbers equal the order of the start times in the files. Table 1 and Table 2 values are
read from Supplementary Table 1, which agrees with the paper's tables. Windows around events are
taken as whole two-photon frames from the trial table's frame numbers, which is a standard
choice. All ROIs in each file are used, as in the paper's code.

What was found about the dataset.

- The session start times are stated with a Pacific offset (-07:00 or -08:00), but their clock
  values are those Supplementary Table 1 gives as UTC. For four sessions this puts the date a day
  after the date in Table 2, and the date of birth implied by the age of mouse 418779 a day
  after Table 1's.
- The three sessions of mouse 433414 are recorded at a depth of 60 um; Tables 2 and S1 give 75 um.
- The files hold no QC result, local session number or stimulus seed. The paper's analyses use
  only sessions that passed QC, and the dandiset's description points to an outside table for
  them.
- The dF/F of the 28 dendritic sessions is stored gzip-chunked in small chunks, which makes
  streaming it through h5py slow; the somatic files store it contiguous.
- The plane (somata or dendrites) is not recorded; it follows from the depth.
- Square size and speed are given in pixels of the unwarped stimulus, without the geometry needed
  to convert them to the degrees the paper states.
- The clock check fails in two sessions (777496949, 15 ROIs; 838633305, failed QC).